# E1 — Constant-load sweep

Question (`IMPLEMENTATION_PLAN.md` §14.1 E1): how do recall and latency change as Poisson load ρ grows, for `static_ef`, `ada_ef`, `laef_uniform` and `laef_wf`?

Headline (§14.2): **`laef_wf` vs `laef_uniform` at matched SLO attainment** — does water-filling keep more recall (mean, P5, P1) than equal cuts for the same latency outcome?

**Input:** `results/e1_load_sweep/summary.csv` (one row per run, written by `scripts/run_experiments.py`; columns = run parameters `dataset, policy, threads, slo_mult, profile, rho, repeat` + every key of the run's `summary.json`). Repeats are averaged; error bars are ± one standard deviation.

In [ ]:
import pathlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = pathlib.Path.cwd().resolve()
while not (ROOT / 'configs').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
EXP = ROOT / 'results' / 'e1_load_sweep'
FIG = ROOT / 'results' / 'figures' / 'e1_load_sweep'
FIG.mkdir(parents=True, exist_ok=True)

POLICIES = ['static_ef', 'ada_ef', 'laef_uniform', 'laef_wf']
COLORS = {'static_ef': 'tab:gray', 'ada_ef': 'tab:blue', 'laef_uniform': 'tab:orange', 'laef_wf': 'tab:green'}
PARAMS = ['dataset', 'policy', 'threads', 'slo_mult', 'rho']
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})

summary_csv = EXP / 'summary.csv'
if not summary_csv.exists():
    raise FileNotFoundError(f'{summary_csv} not found: run  python scripts/run_experiments.py configs/experiments/e1_load_sweep.yaml')
runs = pd.read_csv(summary_csv)
print(len(runs), 'runs;', runs.dataset.nunique(), 'datasets;', sorted(runs.policy.unique()))

METRICS = ['mean_recall', 'p5_recall', 'p1_recall', 'frac_target_recall', 'p50_ms', 'p95_ms', 'p99_ms', 'slo_ms',
           'slo_attainment', 'latency_excess_ms', 'goodput_qps', 'achieved_qps', 'offered_qps', 'mean_ef_ratio',
           'mean_ndis', 'forced_floor_rate', 'floor_mode_query_rate']
METRICS = [m for m in METRICS if m in runs.columns]
agg = runs.groupby(PARAMS)[METRICS].agg(['mean', 'std']).reset_index()
agg.columns = ['_'.join(c).rstrip('_') if c[1] else c[0] for c in agg.columns]
agg['p95_over_slo_mean'] = agg['p95_ms_mean'] / agg['slo_ms_mean']
agg.head()

## 1. Recall, latency and SLO attainment vs load

One row of panels per (dataset, SLO multiplier). The dashed line in the latency panel is p95 = T_SLO.

In [ ]:
PANELS = [('mean_recall', 'mean recall'), ('p5_recall', 'P5 recall (hardest 5%)'),
          ('p95_over_slo', 'p95 latency / T_SLO'), ('slo_attainment', 'SLO attainment'),
          ('mean_ef_ratio', 'mean ef_used / ef_pref')]

def plot_vs_rho(ax, d, metric, label):
    for pol in POLICIES:
        s = d[d.policy == pol].sort_values('rho')
        if s.empty:
            continue
        y = s[f'{metric}_mean']
        err = s.get(f'{metric}_std')
        ax.errorbar(s.rho, y, yerr=err if err is not None else None, marker='o', ms=3, capsize=2,
                    color=COLORS[pol], label=pol)
    ax.set(xlabel='load ρ (× μ_sat)', title=label)

for (ds, m), d in agg.groupby(['dataset', 'slo_mult']):
    fig, axes = plt.subplots(1, len(PANELS), figsize=(4 * len(PANELS), 3.4))
    for ax, (metric, label) in zip(axes, PANELS):
        if f'{metric}_mean' in d:
            plot_vs_rho(ax, d, metric, label)
    axes[2].axhline(1.0, color='k', ls='--', lw=1)
    axes[2].set_yscale('log')
    axes[0].legend(fontsize=7)
    fig.suptitle(f'{ds}   T_SLO = {m:g} × idle S_p95 ({d.slo_ms_mean.iloc[0]:.2f} ms)', y=1.02)
    fig.tight_layout()
    fig.savefig(FIG / f'{ds}_m{m:g}_vs_rho.png', bbox_inches='tight')
    plt.show()

## 2. Throughput: goodput vs offered load

Goodput counts only queries that met the SLO. Every query is served (no drops), so overload shows up as latency, not as missing queries.

In [ ]:
for (ds, m), d in agg.groupby(['dataset', 'slo_mult']):
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
    for pol in POLICIES:
        s = d[d.policy == pol].sort_values('rho')
        if s.empty:
            continue
        ax[0].plot(s.offered_qps_mean, s.goodput_qps_mean, '-o', ms=3, color=COLORS[pol], label=pol)
        ax[1].plot(s.rho, s.floor_mode_query_rate_mean if 'floor_mode_query_rate_mean' in s else np.nan,
                   '-o', ms=3, color=COLORS[pol])
    lim = d.offered_qps_mean.max()
    ax[0].plot([0, lim], [0, lim], 'k:', lw=1)
    ax[0].set(xlabel='offered QPS', ylabel='goodput QPS (within SLO)', title=f'{ds}, m={m:g}')
    ax[1].set(xlabel='load ρ', ylabel='fraction of queries in floor mode', title='floor mode (ef = k, SLO may slip)')
    ax[0].legend(fontsize=7)
    fig.tight_layout()
    fig.savefig(FIG / f'{ds}_m{m:g}_goodput.png')
    plt.show()

## 3. Headline: water-filling vs uniform cut at matched SLO attainment

For each (dataset, m, ρ) both LA-ef variants are compared. A pair counts as *matched* when their SLO attainments differ by at most `MATCH_TOL`; for matched pairs the recall difference is attributable to the cut policy alone (same controller, bucket and floor mode).

In [ ]:
MATCH_TOL = 0.02
key = ['dataset', 'slo_mult', 'rho']
wf = agg[agg.policy == 'laef_wf'].set_index(key)
uc = agg[agg.policy == 'laef_uniform'].set_index(key)
pairs = wf.join(uc, lsuffix='_wf', rsuffix='_uc', how='inner')
head = pd.DataFrame({
    'attain_wf': pairs.slo_attainment_mean_wf,
    'attain_uc': pairs.slo_attainment_mean_uc,
    'matched': (pairs.slo_attainment_mean_wf - pairs.slo_attainment_mean_uc).abs() <= MATCH_TOL,
    'Δ mean recall': pairs.mean_recall_mean_wf - pairs.mean_recall_mean_uc,
    'Δ P5 recall': pairs.p5_recall_mean_wf - pairs.p5_recall_mean_uc,
    'Δ P1 recall': pairs.p1_recall_mean_wf - pairs.p1_recall_mean_uc,
    'Δ p95/SLO': pairs.p95_over_slo_mean_wf - pairs.p95_over_slo_mean_uc,
}).reset_index()
head.to_csv(FIG / 'headline_wf_vs_uniform.csv', index=False)
print('matched pairs:', int(head.matched.sum()), 'of', len(head))
print(head[head.matched][['Δ mean recall', 'Δ P5 recall', 'Δ P1 recall']].describe().round(4))
head.round(4)

In [ ]:
# Recall–attainment frontier: each point is one (m, ρ) operating point; up-and-right is better.
for ds, d in agg.groupby('dataset'):
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
    for pol in POLICIES:
        s = d[d.policy == pol]
        if s.empty:
            continue
        ax[0].scatter(s.slo_attainment_mean, s.mean_recall_mean, color=COLORS[pol], label=pol, s=18)
        ax[1].scatter(s.slo_attainment_mean, s.p5_recall_mean, color=COLORS[pol], s=18)
    ax[0].set(xlabel='SLO attainment', ylabel='mean recall', title=f'{ds}: mean recall')
    ax[1].set(xlabel='SLO attainment', ylabel='P5 recall', title='tail (P5) recall')
    ax[0].legend(fontsize=7)
    fig.tight_layout()
    fig.savefig(FIG / f'{ds}_frontier.png')
    plt.show()

## 4. Compact table for the report

In [ ]:
cols = ['mean_recall_mean', 'p5_recall_mean', 'p1_recall_mean', 'p95_ms_mean', 'slo_ms_mean', 'slo_attainment_mean',
        'goodput_qps_mean', 'mean_ef_ratio_mean']
cols = [c for c in cols if c in agg]
table = agg.pivot_table(index=['dataset', 'slo_mult', 'rho'], columns='policy', values=cols)
table.to_csv(FIG / 'e1_table.csv')
table.round(3)